# Character-Level Text Generation
That is:

It receives JSON packet lines as raw text.

It treats each character as a token.

The RNN learns all character sequences sequentially.

New packets are created as text in the same format.

## STEP 1 — Preparing training data

In [ ]:
import json
import os

data = []

with open('../Datasets/Experiment_1_one_way_communication_10_minute_input_sample.json', 'r') as file:
    for line in file:
        data.append(json.loads(line))

print(data)


[{'No.': 1, 'Time': 0.0, 'Source': '0x1de6', 'Destination': 'Broadcast', 'Protocol': 'ZigBee', 'Length': 80, 'Info': 'Link Status', 'Info_clean': 'Link Status'}, {'No.': 2, 'Time': 8.041278, 'Source': '0x1de6', 'Destination': '0xd7a7', 'Protocol': 'ZigBee HA', 'Length': 69, 'Info': 'ZCL: Read Attributes Response, Seq: 216', 'Info_clean': 'ZCL: Read Attributes Response,'}, {'No.': 3, 'Time': 8.073592, 'Source': '0x1de6', 'Destination': '0xd7a7', 'Protocol': 'ZigBee HA', 'Length': 61, 'Info': 'ZCL: Read Attributes Response, Seq: 217', 'Info_clean': 'ZCL: Read Attributes Response,'}, {'No.': 4, 'Time': 14.242098, 'Source': '0x1de6', 'Destination': 'Broadcast', 'Protocol': 'ZigBee', 'Length': 80, 'Info': 'Link Status', 'Info_clean': 'Link Status'}, {'No.': 5, 'Time': 20.786429, 'Source': '0x1de6', 'Destination': '0xd7a7', 'Protocol': 'ZigBee HA', 'Length': 69, 'Info': 'ZCL: Read Attributes Response, Seq: 239', 'Info_clean': 'ZCL: Read Attributes Response,'}, {'No.': 6, 'Time': 20.812187, '

In [2]:
""" Change Broadcast with the address """
for item in data:
    if item['Destination'] == 'Broadcast':
        item['Destination'] = '0xfffc'

In [3]:
""" Create sample prompt from json file """
Sample_Packets= []

for item in data:
    sample_packets = {
        'time': str(item['Time']),
        'src': item['Source'],
        'dst': item['Destination'],
        'protocol': item['Protocol'],
        'length': str(item['Length']),
        'info': item['Info']
    }
    Sample_Packets.append(sample_packets)


print(Sample_Packets)

[{'time': '0.0', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}, {'time': '8.041278', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '69', 'info': 'ZCL: Read Attributes Response, Seq: 216'}, {'time': '8.073592', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '61', 'info': 'ZCL: Read Attributes Response, Seq: 217'}, {'time': '14.242098', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}, {'time': '20.786429', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '69', 'info': 'ZCL: Read Attributes Response, Seq: 239'}, {'time': '20.812187', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '61', 'info': 'ZCL: Read Attributes Response, Seq: 240'}, {'time': '28.563272', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}, {'time': '33.56803', 'src': '0x1de6', 'dst': '0xd7a7', 'p

In [4]:
# SAVE SAMPLE PACKETS AS JSON FILE
with open("EXP1_sample_packets.txt", "w") as f:
    for pkt in Sample_Packets:
        f.write(json.dumps(pkt) + "\n")

## STEP 2 — Character-level LSTM

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

# ---- LOAD DATA ----
text = open("EXP1_sample_packets.txt").read()
chars = sorted(list(set(text))) # unique characters, creating vocabulary
stoi = {s:i for i,s in enumerate(chars)} # stoi : character → index
itos = {i:s for s,i in stoi.items()} # itos : index → character

def encode(s): return torch.tensor([stoi[c] for c in s], dtype=torch.long) # converts string to numbers
def decode(idx): return ''.join([itos[i] for i in idx]) # recreates string from numbers

data = encode(text) # Converting all characters in the raw text to an integer ID array.

# ---- CREATE SEQUENCES ----
seq_len = 200 # length of each sequence for training
def get_batch(batch_size=32):
    ix = torch.randint(len(data)-seq_len-1, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in ix])
    y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    return x, y

# ---- CREATE MODEL ----
class CharRNN(nn.Module):
    def __init__(self, vocab, hidden=256):
        super().__init__()
        self.embed = nn.Embedding(vocab, 128) #embedding layer
        self.lstm = nn.LSTM(128, hidden, num_layers=2, batch_first=True) # LSTM hidden layer
        self.fc = nn.Linear(hidden, vocab) # fully connected output layer

    def forward(self, x, h=None): # Input x → embedding → LSTM → output logits
        x = self.embed(x)
        out, h = self.lstm(x, h)
        logits = self.fc(out)
        return logits, h

model = CharRNN(len(chars))
opt = optim.Adam(model.parameters(), lr=1e-3)
# opt = optim.Adam(model.parameters(), lr=5e-4)

loss_fn = nn.CrossEntropyLoss() # Cross entropy loss: standard for next character prediction


best_loss = float('inf')
patience = 20
counter = 0

# ---- TRAIN ----
for step in range(3000):
    xb, yb = get_batch() # An input/output array of 32 batches x 200 characters
    logits, _ = model(xb)
    loss = loss_fn(logits.view(-1, len(chars)), yb.view(-1)) # Calculate loss (flatten and compare)

    opt.zero_grad()
    loss.backward()
    opt.step()

    if loss.item() < best_loss:
        best_loss = loss.item()
        counter = 0
    else:
        counter += 1
        if counter >= patience: 
            print(f"Early stopping triggered at training epoch {step}")
            break

    if step % 200 == 0:
        print("step", step, "loss", loss.item())

step 0 loss 3.8974692821502686
step 200 loss 0.18735280632972717
Early stopping triggered at training epoch 256


In [6]:
# SAVE THE MODEL
torch.save(model.state_dict(), "exp1_rnn_model.pth")

# SAVE THE VOCABULARY MAPPINGS

with open("exp1_rnn_vocab.json", "w") as f:
    json.dump({
        "chars": chars,
        "stoi": stoi,
        "itos": itos
    }, f)


## STEP 3 — Packet generation

### 1) Finding the average number of characters from sample packages (most accurate)

In [ ]:
import json
import numpy as np

def average_packet_length(packets):
    lengths = []
    for pkt in packets:
        pkt_str = json.dumps(pkt)   # JSON string formatına dönüştür
        lengths.append(len(pkt_str))
    return np.mean(lengths), min(lengths), max(lengths)

avg_len, min_len, max_len = average_packet_length(Sample_Packets)
print("Average characters per packet:", avg_len)
print("Min packet length:", min_len)
print("Max packet length:", max_len)

num_packets = len(Sample_Packets)
print("Number of packets in sample:", num_packets)

minimum_total_packet_length = num_packets * min_len
print("Estimated minimum characters for real traffic:", minimum_total_packet_length)

average_total_packet_length = num_packets * avg_len
print("Estimated total characters for real traffic:", average_total_packet_length)

maximum_total_packet_length = num_packets * max_len
print("Estimated maximum characters for real traffic:", maximum_total_packet_length)

number_of_char = average_total_packet_length + 1000  # Extra buffer for safety
print("Number of characters to generate is calculated as:", int(number_of_char))


Average characters per packet: 138.5859375
Min packet length: 110
Max packet length: 158
Number of packets in sample: 128
Estimated minimum characters for real traffic: 14080
Estimated total characters for real traffic: 17739.0
Estimated maximum characters for real traffic: 20224
Number of characters to generate is calculated as: 18739


In [8]:
"""
This function has four parameters, all of which are classic RNN settings for text generation. 

PARAMETER 1: START
This parameter tells model what character the text generation should start with.
start="\n"
In this way, the generated text starts on a new line.


PARAMETRE 2: length
This determines how many characters it will generate.

1 character ≠ 1 traffic packet
length = 5000 → 5000 characters


PARAMETRE 3: temperature
Temperature = determines how much creativity/variation you will produce.
High → chaotic
Low → deterministic

PARAMETRE 4: model
This is our LSTM model.

"""

def generate(model, start="\n", length=5000, temperature=0.7):
    model.eval()
    chars_out = [stoi[c] for c in start]
    h = None
    x = torch.tensor([chars_out[-1]]).unsqueeze(0)
    for _ in range(length):
        logits, h = model(x, h)
        logits = logits[:, -1, :] / temperature
        prob = torch.softmax(logits, dim=-1).squeeze()
        ix = torch.multinomial(prob, 1).item()
        chars_out.append(ix)
        x = torch.tensor([[ix]])
    return decode(chars_out)

###  2) Generate and Save RNN output as raw

In [9]:
N = list(range(1,11)) # number of trial

for n in N:

    generated_text = generate(model, length=int(number_of_char))
    print(generated_text)

    raw_file_path = f"..\Generated_Traffic\TXT_files\RNN_Exp1_Trial_{n}_raw_generated_10_minutes.txt"

    with open(raw_file_path, "w", encoding="utf-8") as f:
        f.write(generated_text)

<>:8: SyntaxWarning: invalid escape sequence '\G'
<>:8: SyntaxWarning: invalid escape sequence '\G'
C:\Users\nkelesoglu\AppData\Local\Temp\ipykernel_23172\1423954339.py:8: SyntaxWarning: invalid escape sequence '\G'
  raw_file_path = f"..\Generated_Traffic\TXT_files\RNN_Exp1_Trial_{n}_raw_generated_10_minutes.txt"



"ZigBee HA", "length": "61", "info": "ZCL: Read Httributes Response, Seq: 20"}
{"time": "50.547296", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "61", "info": "ZCL: Read Attributes Response, Seq: 31"}
{"time": "562.351868", "src": "0x1de6", "dst": "0xfffc", "protocol": "ZigBee", "length": "80", "info": "Link Status"}
{"time": "13.378071", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 162"}
{"time": "05.88286", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "61", "info": "ZCL: Read Attributes Response, Seq: 189"}
{"time": "28.013271", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "61", "info": "ZCL: Read Attributes Response, Seq: 238"}
{"time": "53.790955", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 116"}
{"time": "572.960097", "src": "0x1de6", "dst": "0xd7a7", "protoc

## STEP 4 — Open raw output → clean → save as JSON

In [10]:
def clean_rnn_output(raw_text): 
    packets = [] 
    lines = raw_text.split("\n") 

    for line in lines: 
        try: 
            line_fixed = line.strip().rstrip(",") 
            pkt = json.loads(line_fixed) 
            packets.append(pkt) 
        except: 
            continue # half packets are automatically skipped 

    # Time filter: 10 minutes (<= 600 seconds) 
    #packets_10min = [p for p in packets if float(p["time"]) <= 600] 

    return packets, packets_10min


N = list(range(1,11)) # number of trial
packets_10min  = []
for n in N:
    raw_file_path = f"../Generated_Traffic\TXT_files\RNN_Exp1_Trial_{n}_raw_generated_10_minutes.txt"

    # read Raw text file
    with open(raw_file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    packets_all, packets_10min = clean_rnn_output(raw_text)

    # print("Total valid packets parsed:", len(packets_all))
    # print("Valid packets in first 10 min:", len(packets_10min))

    # Save: all valid packages

    clean_all_path = f"../Generated_Traffic\JSON_files\RNN_Exp1_Trial_{n}_generated_10_minutes.json"
    with open(clean_all_path, "w", encoding="utf-8") as f:
        json.dump(packets_all, f, indent=2)



<>:22: SyntaxWarning: invalid escape sequence '\T'
<>:35: SyntaxWarning: invalid escape sequence '\J'
<>:22: SyntaxWarning: invalid escape sequence '\T'
<>:35: SyntaxWarning: invalid escape sequence '\J'
C:\Users\nkelesoglu\AppData\Local\Temp\ipykernel_23172\1216672166.py:22: SyntaxWarning: invalid escape sequence '\T'
  raw_file_path = f"../Generated_Traffic\TXT_files\RNN_Exp1_Trial_{n}_raw_generated_10_minutes.txt"
C:\Users\nkelesoglu\AppData\Local\Temp\ipykernel_23172\1216672166.py:35: SyntaxWarning: invalid escape sequence '\J'
  clean_all_path = f"../Generated_Traffic\JSON_files\RNN_Exp1_Trial_{n}_generated_10_minutes.json"
